### 00 data_injest/load_geojson.py

''''
This script loads, preprocesses, and merges location data from a Feather file and a GeoJSON file. Here's a concise overview of the process:

1. Loading and Preprocessing Location Data
Input: A Feather file with location mappings.
Process:
    Load the Feather file into a DataFrame.
    Use a dictionary to split composite Local Government Area (LGA) names into individual entries.
    Standardize names by converting to lowercase and replacing specific substrings (e.g., 'yenagoa' becomes 'yenegoa').
    Clean data by removing unwanted entries and duplicates.
Output: A cleaned DataFrame with standardized state and LGA information.  

2. Loading and Preprocessing GeoJSON Data
Input: A GeoJSON file with LGA boundaries.
Process:
    Load the GeoJSON file into a GeoDataFrame.
    Standardize state and LGA names by converting to lowercase and replacing substrings for consistency.
Output: A preprocessed GeoDataFrame ready for merging.  

3. Merging Data
Input: Cleaned location DataFrame and preprocessed GeoDataFrame.
Process: Merge the DataFrames on state and LGA names to combine location data with geographic boundaries.
Output: A merged DataFrame with an indicator column showing the merge results.  


Example Workflow  
Splitting Example:  
Input: {'Abuja': {'Kuje/Gwagwalada/Abaji': ['kuje', 'gwagwalada', 'abaji']}}  
Output: Separate rows for each LGA name under the specified state.  

Renaming Example:  
Original: ifako ijaiye  
Standardized: ifako ijaye  
'''

In [ ]:
# pip install 'snowflake-connector-python[pandas]' # Fix UserWarning: You have an incompatible version of 'pyarrow' installed (19.0.1)

In [ ]:
import pickle
import geopandas as gpd
from shapely.geometry import Point, Polygon
from sqlalchemy import create_engine, text
import pandas as pd # For stock points and potentially LGA data
from codebase.database.get_connection import get_connection_string, get_connection
from codebase.utils.utils import setup_logging 
from codebase.database.create_connection import create_db_connection_engine

In [ ]:
logger = setup_logging(log_dir='log-sandbox-sp-clustering-and-routing', 
                       projname='log-sandbox-sp-clustering-and-routing')

In [ ]:
# engine = create_db_connection_engine(db_database='VConnectMasterDWR', db_type='replica', logger=logger)

In [ ]:
df_sp_location_mapping = pd.read_feather('./input/df_sp_location_mapping.feather')

In [ ]:
df_sp_location_mapping.head(1)

In [ ]:
def load_and_preprocess_sp_mapping_data():
    try:
        # Load the Feather file
        df_sp_location_mapping = pd.read_feather('./input/df_sp_location_mapping.feather')
        logger.info("Loaded location mapping data from Feather file.")

        # Dictionary for splitting state and LGA names
        dict_split_state_lga = {'Abuja': {'Kuje/Gwagwalada/Abaji': ['kuje', 'gwagwalada', 'abaji']}}

        # Create DataFrame from dictionary
        df_split_state_lga = pd.DataFrame([
            {'State_Name': state_name, 'LGA_Name': lga_name, 'lga_name_split': item}
            for state_name, lga_dict in dict_split_state_lga.items()
            for lga_name, lga_list in lga_dict.items()
            for item in lga_list
        ])

        # Preprocessing pipeline
        df_dist_state_lga = (
            df_sp_location_mapping
            .merge(df_split_state_lga, on=['State_Name', 'LGA_Name'], how='left')
            .assign(state_=lambda df: df['State_Name'].str.lower())
            .assign(lga_=lambda df: df['LGA_Name'].str.lower())
            .assign(lga_=lambda df: df.apply(
                lambda row: row['lga_'] if pd.isna(row['lga_name_split']) else row['lga_name_split'], axis=1
            ))
            .assign(lga_=lambda df: df['lga_'].str.replace('/', ' '))
            .assign(lga_=lambda df: df['lga_'].replace({
                'yenagoa': 'yenegoa',
                'ifako ijaiye': 'ifako ijaye',
                'sagamu': 'shagamu',
                'garun mallam': 'garun malam',
                'amac 1': 'municipal area council',
                'kachako': 'takai',
                'mbaitoli': 'mbatoli'
            }))
            [['State_Name', 'State_ID', 'LGA_Name', 'LGA_ID', 'state_', 'lga_']]
            .drop_duplicates()
            .query('~lga_.str.contains("self|push")', engine='python')
        )

        logger.info("Preprocessing completed successfully.")
        logger.info(f"Initial length: {len(df_sp_location_mapping)}")
        logger.info(f"Processed length: {len(df_dist_state_lga)}")
        logger.info(f"Columns: {df_dist_state_lga.columns.tolist()}")

        return df_dist_state_lga

    except FileNotFoundError:
        logger.error('The specified Feather file was not found.')
    except Exception as e:
        logger.error(f'An error occurred: {e}')


def load_and_preprocess_sp_lcda_mapping_data():
    try:
        # Load the Feather file
        df_sp_location_mapping = pd.read_feather('./input/df_sp_location_mapping.feather')
        logger.info("Loaded location mapping data from Feather file.")

        # Dictionary for splitting state and LGA names
        dict_split_state_lga = {'Abuja': {'Kuje/Gwagwalada/Abaji': ['kuje', 'gwagwalada', 'abaji']}}

        # Create DataFrame from dictionary
        df_split_state_lga = pd.DataFrame([
            {'State_Name': state_name, 'LGA_Name': lga_name, 'lga_name_split': item}
            for state_name, lga_dict in dict_split_state_lga.items()
            for lga_name, lga_list in lga_dict.items()
            for item in lga_list
        ])

        # Preprocessing pipeline
        df_dist_state_lga = (
            df_sp_location_mapping
            .merge(df_split_state_lga, on=['State_Name', 'LGA_Name'], how='left')
            .assign(state_=lambda df: df['State_Name'].str.lower())
            .assign(lga_=lambda df: df['LGA_Name'].str.lower())
            ## LCDA preprocessing
            .assign(lcda_=lambda df: df['LCDA_Name'].str.lower())
            .assign(lga_=lambda df: df['lga_'].str.replace('/', ' '))
            ## LGA preprocessing
            .assign(lga_=lambda df: df.apply(
                lambda row: row['lga_'] if pd.isna(row['lga_name_split']) else row['lga_name_split'], axis=1
            ))
            .assign(lga_=lambda df: df['lga_'].str.replace('/', ' '))
            .assign(lga_=lambda df: df['lga_'].replace({
                'yenagoa': 'yenegoa',
                'ifako ijaiye': 'ifako ijaye',
                'sagamu': 'shagamu',
                'garun mallam': 'garun malam',
                'amac 1': 'municipal area council',
                'kachako': 'takai',
                'mbaitoli': 'mbatoli'
            }))
            [['State_Name', 'State_ID', 'LGA_Name', 'LGA_ID', 'LCDA_Name', 'LCDA_ID', 'state_', 'lga_', 'lcda_']]
            .drop_duplicates()
            .query('~lga_.str.contains("self|push")', engine='python')
        )

        logger.info("Preprocessing completed successfully.")
        logger.info(f"Initial length: {len(df_sp_location_mapping)}")
        logger.info(f"Processed length: {len(df_dist_state_lga)}")
        logger.info(f"Columns: {df_dist_state_lga.columns.tolist()}")

        return df_dist_state_lga

    except FileNotFoundError:
        logger.error('The specified Feather file was not found.')
    except Exception as e:
        logger.error(f'An error occurred: {e}')


def load_and_preprocess_lga_geojson():
    lgas_geojson_path = '../input/geojson/GRID3_NGA_-_Operational_LGA_Boundaries.geojson'
    try:
        lgas_gdf = gpd.read_file(lgas_geojson_path)
        logger.info(f"Loaded {len(lgas_gdf)} LGAs from GeoJSON.")
        drop_cols = [ 'FID','globalid', 'uniq_id', 'timestamp', 'editor']
        # Preprocessing for easy merging
        lgas_gdf = (
            lgas_gdf
            .assign(state_=lambda df: df['statename'].str.lower())
            .assign(state_=lambda df: df['state_'].replace({'fct': 'abuja'}))
            .assign(lga_=lambda df: df['lganame'].str.lower())
            .assign(lga_=lambda df: df['lga_'].str.replace('/', ' ').str.replace('-', ' '))
            .drop(columns=drop_cols)
        )
        geometry_cols = ['state_', 'lga_','Shape__Area', 'Shape__Length', 'geometry']
        lgas_gdf.columns = [col+"_ng"  if col not in geometry_cols else col for col in lgas_gdf.columns]
        
        lgas_gdf_proj = lgas_gdf.to_crs("EPSG:6933")  # WGS 84 / World Cylindrical Equal Area
        lgas_gdf['area_km2'] = (lgas_gdf_proj.geometry.area / 1_000_000).round(4)  # Convert m^2 to km^2
        
        lgas_gdf['geometry_wkt'] = [geo.wkt for geo in lgas_gdf.geometry] # Convert shapely geometry to WKT
        
        return lgas_gdf

    except FileNotFoundError:
        logger.error('The specified GeoJSON file was not found.')
    except Exception as e:
        logger.error(f'An error occurred: {e}')


def load_and_preprocess_lcda_geojson(lcda_geojson_path = '../input/geojson/Nigeria_-_Ward_Boundaries.geojson'):
    
    try:
        lcdas_gdf = gpd.read_file(lcda_geojson_path)
        logger.info(f"Loaded {len(lcdas_gdf)} LGAs from GeoJSON.")
        drop_cols = [ 'FID','globalid', 'uniq_id', 'timestamp', 'editor']
        # Preprocessing for easy merging
        lcdas_gdf = (
            lcdas_gdf
            .assign(state_=lambda df: df['statename'].str.lower())
            .assign(state_=lambda df: df['state_'].replace({'fct': 'abuja'}))
            .assign(lga_=lambda df: df['lganame'].str.lower())
            .assign(lcda_=lambda df: df['wardname'].str.lower())
            .assign(lga_=lambda df: df['lga_'].str.replace('/', ' ').str.replace('-', ' ')) 
            .assign(lcda_=lambda df: df['lcda_'].str.replace('/', ' ').str.replace('-', ' ')) 
            .drop(columns=drop_cols) 
        )
        geometry_cols = ['state_', 'lga_','lcda_', 'Shape__Area', 'Shape__Length', 'geometry']
        lcdas_gdf.columns = [col+"_ng"  if col not in geometry_cols else col for col in lcdas_gdf.columns]
        
        lcdas_gdf_proj = lcdas_gdf.to_crs("EPSG:6933")  # WGS 84 / World Cylindrical Equal Area
        lcdas_gdf['area_km2'] = (lcdas_gdf_proj.geometry.area / 1_000_000).round(4)  # Convert m^2 to km^2
        
        # lcdas_gdf['geometry_wkt'] = [geo.wkt for geo in lcdas_gdf.geometry] # Convert shapely geometry to WKT
        
        return lcdas_gdf

    except FileNotFoundError:
        logger.error('The specified GeoJSON file was not found.')
    except Exception as e:
        logger.error(f'An error occurred: {e}')


def merge_data(df_dist_state_lga, lgas_gdf):
    col_join = ['state_', 'lga_']
    df_state_lga_merge_gdf = df_dist_state_lga.merge(lgas_gdf, how='left', on=col_join, indicator=True)
    logger.info("Merge results:\n" + str(df_state_lga_merge_gdf.value_counts('_merge')))
    return df_state_lga_merge_gdf


In [ ]:
# Execute the functions
df_mfc_mapped_lga = load_and_preprocess_sp_mapping_data()
df_mfc_mapped_lcda = load_and_preprocess_sp_lcda_mapping_data()
lgas_gdf = load_and_preprocess_lga_geojson()
lcda_gdf = load_and_preprocess_lcda_geojson()
df_state_lga_gdf = merge_data(df_mfc_mapped_lga, lgas_gdf) 

with open('./input/df_state_lga_gdf.feather', 'wb') as filename:
    pickle.dump(obj=df_state_lga_gdf, file=filename) 

In [ ]:
# lcda_gdf.head(1)

In [ ]:
# df_dist_state_lga.columns
# pd.read_feather('./input/df_sp_location_mapping.feather').columns

In [ ]:
col_join = ['state_', 'lga_','lcda_']
df_mapped_lga_lcda_gdf = df_mfc_mapped_lcda.merge(lcda_gdf, how='left', on=col_join, indicator=True)
logger.info("Merge results:\n" + str(df_mapped_lga_lcda_gdf.value_counts('_merge'))) 

In [ ]:
lcda_gdf.columns

In [ ]:
cols_map = ['LCDA_ID', 'state_', 'lga_', 'lcda_']
cols_gdf = ['state_', 'lga_', 'lcda_', 'wardname_ng', 'wardcode_ng']

In [ ]:
df_issue = df_mapped_lga_lcda_gdf.query('_merge == "left_only"').reset_index(drop=True)

df_issue_unq_dict = df_issue[['state_', 'lga_']].drop_duplicates().sort_values('state_').reset_index(drop=True).to_dict(orient='records')
df_issue_unq_dict[0]


In [ ]:
indx = 0
 
print(
    lcda_gdf.query('state_ == @df_issue_unq_dict[@indx]["state_"] and lga_ == @df_issue_unq_dict[@indx]["lga_"]')[cols_gdf].reset_index(drop=True)
)
 
# print(
#     df_issue.query('state_ == @df_issue_unq_dict[@indx]["state_"] and lga_ == @df_issue_unq_dict[@indx]["lga_"]')[cols_map].reset_index(drop=True)
# )



In [ ]:
# pip install sentence-transformers 

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Sample data
mapped_data = pd.DataFrame({
    'LCDA_ID': [309, 499, 500, 501, 502, 503, 504, 505, 506, 512, 514, 515, 485, 486, 487, 488, 496, 517],
    'state_': ['abuja'] * 18,
    'lga_': ['municipal area council'] * 18,
    'lcda_': ['dei dei', 'wuse 2', 'wuse zone 1', 'wuse zone 2', 'wuse zone 3', 'wuse zone 4',
              'wuse zone 5', 'wuse zone 6', 'wuse zone 7', 'lugbe', 'sauka',
              'airport road', 'kado federal housing', 'mabushi', 'utako', 'jabi',
              'ministers hill', 'idu']
})

official_data = pd.DataFrame({
    'state_': ['abuja'] * 12,
    'lga_': ['municipal area council'] * 12,
    'lcda_': ['wuse', 'jiwa', 'gwagwa', 'karshi 1', 'garki 1', 'nyanya 1',
              'orozo', 'city center 1', 'karu', 'gui', 'gwarinpa', 'kabusa'],
    'wardname_ng': ['Wuse', 'Jiwa', 'Gwagwa', 'Karshi 1', 'Garki 1', 'Nyanya 1',
                   'Orozo', 'City Center 1', 'Karu', 'Gui', 'Gwarinpa', 'Kabusa'],
    'wardcode_ng': ['FCTABC12', 'FCTABC06', 'FCTABC04', 'FCTABC08', 'FCTABC02',
                    'FCTABC10', 'FCTABC11', 'FCTABC01', 'FCTABC09', 'FCTABC03',
                    'FCTABC05', 'FCTABC07']
})

# Load a pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
mapped_embeddings = model.encode(mapped_data['lcda_'].tolist())
official_embeddings = model.encode(official_data['lcda_'].tolist())

# Calculate cosine similarity
similarity_matrix = cosine_similarity(mapped_embeddings, official_embeddings)

# Find the best match for each row in mapped_data
best_match_indices = similarity_matrix.argmax(axis=1)

# Extract matched wardname and wardcode using the best match indices
mapped_data['wardname_ng'] = mapped_data.index.map(
    lambda x: official_data.loc[best_match_indices[x], 'wardname_ng']
)
mapped_data['wardcode_ng'] = mapped_data.index.map(
    lambda x: official_data.loc[best_match_indices[x], 'wardcode_ng']
)

# Display the matched data
print(mapped_data[['state_', 'lga_', 'lcda_', 'wardname_ng', 'wardcode_ng']])


/home/bt/project/demand_engine/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   state_                    lga_                 lcda_    wardname_ng  \
0   abuja  municipal area council               dei dei  City Center 1   
1   abuja  municipal area council                wuse 2           Wuse   
2   abuja  municipal area council           wuse zone 1           Wuse   
3   abuja  municipal area council           wuse zone 2           Wuse   
4   abuja  municipal area council           wuse zone 3           Wuse   
5   abuja  municipal area council           wuse zone 4           Wuse   
6   abuja  municipal area council           wuse zone 5           Wuse   
7   abuja  municipal area council           wuse zone 6           Wuse   
8   abuja  municipal area council           wuse zone 7           Wuse   
9   abuja  municipal area council                 lugbe         Kabusa   
10  abuja  municipal area council                 sauka           Jiwa   
11  abuja  municipal area council          airport road  City Center 1   
12  abuja  municipal area council  kad

In [ ]:
# df_unmatch = df_state_lga_merge_gdf.query('_merge != "both"').sort_values('state_')
# print(len(df_unmatch))# nasarawa	amac 3
# df_unmatch 

In [ ]:
# df_state_lga_gdf.head(1)
df_state_lga_gdf.columns

## Export Data to DB

In [ ]:
## Data Export 
df_ds_gis_lga = df_state_lga_gdf[[ 'LGA_ID','lgacode_ng', 'LGA_Name', 'lganame_ng', 'State_ID', 
                                  'State_Name', 'statename_ng', 'geometry', 'area_km2']].reset_index(drop=True)
df_ds_gis_lga['population_density'] = 0
df_ds_gis_lga['geometry'] = df_ds_gis_lga['geometry'].apply(lambda x: f"geometry::STGeomFromText('{x}', 4326)")


dict_cols = {'LGA_ID': 'lga_id',
            'lgacode_ng': 'lga_code',
            'LGA_Name': 'lga_name',
            'lganame_ng': 'lga_name_ng',
            'State_ID': 'state_id',
            'State_Name': 'state_name',
            'statename_ng': 'state_name_ng',
            # 'geometry_wkt': 'geometry',
            'area_sq_km': 'area_km2',
            'population_density': 'population_density'}

df_ds_gis_lga.rename(columns=dict_cols, inplace=True)

In [ ]:
# !pip install geoalchemy2

In [ ]:
from sqlalchemy import create_engine, MetaData, Table, text
from sqlalchemy.orm import sessionmaker
from sqlalchemy.exc import SQLAlchemyError

def upsert_dataframe_sqlalchemy(df, table_name, engine, match_cols, update_cols):
    """
    Upsert DataFrame using SQLAlchemy with SQL Server.

    Args:
        df: DataFrame to upsert
        table_name: Target table name
        engine: SQLAlchemy engine
        match_cols: Columns to match on for upsert
        update_cols: Columns to update when matched
    """
    metadata = MetaData()
    table = Table(table_name, metadata, autoload_with=engine)

    # Create a session
    Session = sessionmaker(bind=engine)
    session = Session()

    try:
        for _, row in df.iterrows():
            # Create a dictionary of the row data
            row_data = row.to_dict()

            # Create the MERGE statement
            merge_stmt = text(f"""
                MERGE INTO {table_name} AS target
                USING (VALUES ({', '.join([f":{col}" for col in row_data])})) AS source ({', '.join(row_data.keys())})
                ON {' AND '.join([f"target.{col} = source.{col}" for col in match_cols])}
                WHEN MATCHED THEN
                    UPDATE SET {', '.join([f"{col} = source.{col}" for col in update_cols])}
                WHEN NOT MATCHED THEN
                    INSERT ({', '.join(row_data.keys())})
                    VALUES ({', '.join([f":{col}" for col in row_data])});
            """)

            # Execute the statement with the row data
            session.execute(merge_stmt, row_data)

        session.commit()
        print(f"Successfully upserted {len(df)} rows to {table_name}")

    except SQLAlchemyError as e:
        session.rollback()
        print(f"Upsert failed for {table_name}: {e}")
        raise

    finally:
        session.close()


In [ ]:
df_ds_gis_lga.info()

In [ ]:
# SQL Server Spatial Data Export for Nigerian LGAs
import logging
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import pyodbc

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def export_lga_data_to_sqlserver(df_state_lga_gdf, engine):
    """
    Export LGA geodata to SQL Server with spatial geometry support
    
    Args:
        df_state_lga_gdf: GeoDataFrame with LGA data
        engine: SQLAlchemy engine connection to SQL Server
    """
    
    try:
        # Data selection and preparation
        df_ds_gis_lga = df_state_lga_gdf[[ 
            'LGA_ID', 'lgacode_ng', 'LGA_Name', 'lganame_ng', 
            'State_ID', 'State_Name', 'statename_ng', 'geometry', 'area_km2'
        ]].reset_index(drop=True).copy()
        
        # Add placeholder for population density
        df_ds_gis_lga['population_density'] = 0.0
        
        # Column mapping - corrected for your schema
        dict_cols = {
            'LGA_ID': 'lga_id',
            'lgacode_ng': 'lgacode',  # Note: your schema expects INT, not lga_code
            'LGA_Name': 'lga_name',
            'lganame_ng': 'lga_name_ng',
            'State_ID': 'state_id',
            'State_Name': 'state_name',
            'statename_ng': 'state_name_ng',  # Note: missing from your schema
            'area_km2': 'area_km2',
            'population_density': 'population_density'
        }
        
        # Rename columns (except geometry - we'll handle that separately)
        df_export = df_ds_gis_lga.drop('geometry', axis=1).rename(columns=dict_cols)
        
        # Add geometry as WKT string for SQL Server processing
        df_export['geometry_wkt'] = df_ds_gis_lga['geometry'].apply(
            lambda x: x.wkt if x is not None else None
        )
        
        # Remove statename_ng if not in your actual schema
        if 'state_name_ng' in df_export.columns:
            df_export = df_export.drop('state_name_ng', axis=1)
        
        logger.info(f"Exporting {len(df_export)} LGA records to SQL Server")
        logger.info(f"Columns: {list(df_export.columns)}")
        
        # Method 1: Using raw SQL for better control (recommended for SQL Server spatial)
        export_with_raw_sql(df_export, engine)
        
    except Exception as e:
        logger.error(f"Critical error during data export: {str(e)}")
        raise

def export_with_raw_sql(df_export, engine):
    """
    Export using raw SQL with SQL Server spatial functions
    """
    
    # SQL template for insertion with SQL Server spatial syntax
    insert_sql = """
    INSERT INTO ds_gis_lgas (
        lga_id, lgacode, lga_name, lga_name_ng, 
        state_id, state_name, geometry, area_km2, population_density
    ) VALUES (
        ?, ?, ?, ?, ?, ?, 
        geometry::STGeomFromText(?, 4326), ?, ?
    )
    """
    
    success_count = 0
    error_count = 0
    
    with engine.connect() as conn:
        # Process in chunks for better memory management
        chunk_size = 1000
        total_rows = len(df_export)
        
        for i in range(0, total_rows, chunk_size):
            chunk = df_export.iloc[i:i + chunk_size]
            chunk_num = i // chunk_size + 1
            total_chunks = (total_rows + chunk_size - 1) // chunk_size
            
            logger.info(f"Processing chunk {chunk_num}/{total_chunks} ({len(chunk)} rows)")
            
            # Prepare batch data
            batch_data = []
            for _, row in chunk.iterrows():
                try:
                    batch_data.append((
                        int(row['lga_id']) if pd.notna(row['lga_id']) else None,
                        int(row['lgacode']) if pd.notna(row['lgacode']) else None,
                        str(row['lga_name']) if pd.notna(row['lga_name']) else None,
                        str(row['lga_name_ng']) if pd.notna(row['lga_name_ng']) else None,
                        int(row['state_id']) if pd.notna(row['state_id']) else None,
                        str(row['state_name']) if pd.notna(row['state_name']) else None,
                        str(row['geometry_wkt']) if pd.notna(row['geometry_wkt']) else None,
                        float(row['area_km2']) if pd.notna(row['area_km2']) else None,
                        float(row['population_density']) if pd.notna(row['population_density']) else 0.0
                    ))
                except Exception as row_error:
                    logger.error(f"Error preparing row data: {str(row_error)}")
                    error_count += 1
            
            # Execute batch insert
            try:
                trans = conn.begin()
                conn.execute(text(insert_sql), batch_data)
                trans.commit()
                success_count += len(batch_data)
                logger.info(f"Successfully inserted chunk {chunk_num} ({len(batch_data)} rows)")
                
            except SQLAlchemyError as e:
                trans.rollback()
                logger.error(f"Error inserting chunk {chunk_num}: {str(e)}")
                
                # Fall back to row-by-row insertion for this chunk
                for row_data in batch_data:
                    try:
                        row_trans = conn.begin()
                        conn.execute(text(insert_sql), [row_data])
                        row_trans.commit()
                        success_count += 1
                    except Exception as row_error:
                        row_trans.rollback()
                        logger.error(f"Failed to insert individual row: {str(row_error)}")
                        error_count += 1
    
    logger.info(f"Export completed: {success_count} successful, {error_count} errors")

def export_with_pandas_to_sql(df_export, engine):
    """
    Alternative method using pandas to_sql (may have limitations with geometry)
    """
    
    # Separate geometry and non-geometry data
    df_no_geom = df_export.drop('geometry_wkt', axis=1)
    
    try:
        # First insert non-geometry data
        df_no_geom.to_sql(
            'ds_gis_lgas_temp',  # Temporary table
            engine,
            if_exists='replace',
            index=False,
            chunksize=1000
        )
        
        # Then update with geometry using SQL
        update_geometry_sql = """
        UPDATE lga SET geometry = geometry::STGeomFromText(temp.geometry_wkt, 4326)
        FROM ds_gis_lgas lga
        INNER JOIN ds_gis_lgas_temp temp ON lga.lga_id = temp.lga_id
        """
        
        with engine.connect() as conn:
            conn.execute(text(update_geometry_sql))
            # Drop temporary table
            conn.execute(text("DROP TABLE ds_gis_lgas_temp"))
            
    except Exception as e:
        logger.error(f"Error in pandas to_sql method: {str(e)}")
        raise

def validate_data_before_export(df_state_lga_gdf):
    """
    Validate data before export to catch common issues
    """
    
    logger.info("Validating data before export...")
    
    # Check for missing geometries
    missing_geom = df_state_lga_gdf[df_state_lga_gdf['geometry'].isna()]
    if not missing_geom.empty:
        logger.warning(f"Found {len(missing_geom)} records with missing geometry")
    
    # Check for invalid geometries - correct way
    try:
        # Apply is_valid to each geometry object
        valid_mask = df_state_lga_gdf['geometry'].apply(
            lambda geom: geom.is_valid if geom is not None else False
        )
        invalid_geom = df_state_lga_gdf[~valid_mask]
        if not invalid_geom.empty:
            logger.warning(f"Found {len(invalid_geom)} records with invalid geometry")
            # Log some details about invalid geometries
            for idx, row in invalid_geom.head(5).iterrows():
                geom = row['geometry']
                if geom is not None:
                    logger.warning(f"Invalid geometry at index {idx}: {geom.geom_type}, "
                                 f"LGA: {row.get('LGA_Name', 'Unknown')}")
    except Exception as e:
        logger.error(f"Error checking geometry validity: {str(e)}")
        # Alternative check using try/except for each geometry
        invalid_count = 0
        for idx, geom in df_state_lga_gdf['geometry'].items():
            if geom is not None:
                try:
                    _ = geom.wkt  # Try to access WKT - will fail for invalid geometries
                except Exception:
                    invalid_count += 1
        if invalid_count > 0:
            logger.warning(f"Found approximately {invalid_count} problematic geometries")
    
    # Check for duplicate LGA IDs
    duplicate_ids = df_state_lga_gdf[df_state_lga_gdf.duplicated('LGA_ID', keep=False)]
    if not duplicate_ids.empty:
        logger.warning(f"Found {len(duplicate_ids)} records with duplicate LGA_ID")
    
    # Check for empty geometries
    empty_geom = df_state_lga_gdf[
        df_state_lga_gdf['geometry'].apply(
            lambda geom: geom.is_empty if geom is not None else True
        )
    ]
    if not empty_geom.empty:
        logger.warning(f"Found {len(empty_geom)} records with empty geometry")
    
    # Check data types and basic stats
    logger.info(f"Data types:\n{df_state_lga_gdf.dtypes}")
    logger.info(f"Shape: {df_state_lga_gdf.shape}")
    
    # Check area values
    if 'area_km2' in df_state_lga_gdf.columns:
        area_stats = df_state_lga_gdf['area_km2'].describe()
        logger.info(f"Area statistics:\n{area_stats}")
        
        # Check for suspicious area values
        zero_area = df_state_lga_gdf[df_state_lga_gdf['area_km2'] <= 0]
        if not zero_area.empty:
            logger.warning(f"Found {len(zero_area)} records with zero or negative area")
    
    return True

# Usage example with SQL Server connection
def main_export_process(df_state_lga_gdf, connection_string):
    """
    Main export process with proper SQL Server connection
    """
    
    # Validate data first
    validate_data_before_export(df_state_lga_gdf)
    
    # Create SQL Server engine
    engine = create_engine(
        f"mssql+pyodbc:///?odbc_connect={connection_string}",
        echo=False  # Set to True for SQL debugging
    )
    
    # Export data
    export_lga_data_to_sqlserver(df_state_lga_gdf, engine)
    
    logger.info("Export process completed")

# Example connection string format:
# connection_string = "DRIVER={ODBC Driver 17 for SQL Server};SERVER=your_server;DATABASE=your_db;UID=your_user;PWD=your_password"

# Or for Windows Authentication:
# connection_string = "DRIVER={ODBC Driver 17 for SQL Server};SERVER=your_server;DATABASE=your_db;Trusted_Connection=yes"

In [ ]:
# validate_data_before_export(df_state_lga_gdf.head(1))
export_lga_data_to_sqlserver(df_state_lga_gdf, engine)

In [ ]:

# geometry::STGeomFromText('POLYGON ((3.8801563506351 7.34938948333803, 
# ProgrammingError: ('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]'ST_GeomFromEWKT' 
#                    is not a recognized built-in function name. (195) (SQLExecDirectW)")

In [ ]:
# match_cols = ['state_id', 'lga_id']
# update_cols = [col for col in df_ds_gis_lga.columns if col not in match_cols]

# upsert_dataframe_sqlalchemy(df=df_ds_gis_lga, table_name='ds_gis_lgas', 
#                             engine=engine, match_cols=match_cols, update_cols=update_cols)

--------------